# 04. Hyperparameter Optimization & Final Model Evaluation — CampusWell RiskLab

## Section 1: Optimization Objectives & Search Methodology

Welcome to the **CampusWell RiskLab** Hyperparameter Optimization and Final Model Evaluation notebook.

### The Coarse-to-Fine Optimization Strategy
Rather than arbitrary trial-and-error, we execute a systematic two-stage search strategy on the training data ($N=1,600$):

1. **Stage 1 — Broad Exploration (`RandomizedSearchCV`):**
   Explores broad hyperparameter distributions across regularization strength $C$, tree depth, learning rate, and solver algorithms to identify promising convex regions in parameter space.
2. **Stage 2 — Focused Refinement (`GridSearchCV`):**
   Narrows search to a fine grid around the best parameter neighborhood identified in Stage 1 to pinpoint optimal convergence points.

### Objective Function & Constraint
All cross-validation maximizes **Weighted F1-score** under Stratified 5-Fold Cross-Validation:
$$\text{Maximize } F_1^{\text{weighted}} = \sum_{c \in \{\text{Low}, \text{Medium}, \text{High}\}} \frac{N_c}{N} F_{1, c}$$
The final champion model is also subjected to a clinical safety check: zero students with `High` risk should be misdiagnosed as `Low` risk.

## Section 2: Environment Setup & Data Loading

We import project configuration parameters, serialization utilities, metrics computation functions, and search modules.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

from src.config import (
    RANDOM_STATE,
    CV_FOLDS,
    TARGET_CLASSES,
    MODEL_PATH,
    MODEL_METADATA_PATH,
    FINAL_METRICS_PATH,
    OPTIMIZATION_RESULTS_PATH,
    REPORTS_FIGURES_DIR
)
from src.features.preprocessing import build_model_pipeline
from src.steps.preprocessing import load_splits
from src.steps.optimization import (
    run_randomized_search,
    run_grid_search,
    compare_with_baseline,
    select_production_candidate
)
from src.steps.evaluation import (
    train_final_pipeline,
    evaluate_on_test_set,
    analyze_errors,
    generate_evaluation_figures
)
from src.utils.serialization import save_pipeline, save_metadata
from src.utils.metrics import save_metrics, load_metrics

print("Optimization and evaluation environment successfully initialized.")

/home/badr-eddine-slaoui/Documents/CAMPUSWELL-RISΚLAB/models/production_pipeline.joblib
Pipeline(steps=[('feature_engineering', FeatureEngineer()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Year_of_Study',
                                                   'Sleep_Hours',
                                                   'Study_Hours_Per_Week',
                                                   'GPA', 'Social_Media_Hours',
                                                   'Screen_Time_Hours',
                                           

## Section 3: Training and Test Split Ingestion

We ingest the training partition ($N=1,600$) used for hyperparameter cross-validation, and the untouched test partition ($N=400$) reserved exclusively for final evaluation.

In [2]:
X_train, X_test, y_train, y_test = load_splits()
print(f"Training split samples: {len(X_train)} (Features: {X_train.shape[1]})")
print(f"Test split samples:     {len(X_test)} (Features: {X_test.shape[1]})")
print("\nTarget class counts in X_train:")
print(y_train.value_counts())
print("\nTarget class counts in X_test:")
print(y_test.value_counts())

Training split samples: 1600 (Features: 15)
Test split samples:     400 (Features: 15)

Target class counts in X_train:
Mental_Health_Risk
High      544
Medium    528
Low       528
Name: count, dtype: int64

Target class counts in X_test:
Mental_Health_Risk
High      136
Medium    132
Low       132
Name: count, dtype: int64


## Section 4: Hyperparameter Optimization Execution

We optimize the top two candidate architectures identified in the baseline benchmarking phase:
1. **Candidate 1 — Logistic Regression (Top Performer):**
   Hyperparameters tuned: $L_2$ inverse regularization parameter $C \in [0.01, 50.0]$, solver $\in [\text{'lbfgs'}, \text{'saga'}]$, and max iterations.
2. **Candidate 2 — HistGradientBoosting (Runner-Up Ensemble):**
   Hyperparameters tuned: learning rate $\eta \in [0.01, 0.2]$, max leaf nodes $\in [15, 63]$, min samples per leaf $\in [10, 30]$, max iterations, and $L_2$ regularization.

In [3]:
candidates = ["Logistic Regression", "HistGradientBoosting"]
optimization_results = {}

for name in candidates:
    print(f"--- Optimizing {name} ---")
    rs = run_randomized_search(name, X_train, y_train, n_iter=10)
    gs = run_grid_search(name, X_train, y_train, rs.best_params_)
    optimization_results[name] = {
        "random_search_best_score": float(rs.best_score_),
        "best_cv_score": float(gs.best_score_),
        "best_params": gs.best_params_
    }
    print(f"  -> {name} Best CV Weighted F1: {gs.best_score_:.4f}")
    print(f"  -> {name} Optimal Parameters: {gs.best_params_}")

--- Optimizing Logistic Regression ---


  -> Logistic Regression Best CV Weighted F1: 0.7563
  -> Logistic Regression Optimal Parameters: {'classifier__C': 0.75, 'classifier__max_iter': 500, 'classifier__solver': 'saga'}
--- Optimizing HistGradientBoosting ---


  -> HistGradientBoosting Best CV Weighted F1: 0.6876
  -> HistGradientBoosting Optimal Parameters: {'classifier__l2_regularization': 10.0, 'classifier__learning_rate': 0.05, 'classifier__max_iter': 200, 'classifier__max_leaf_nodes': 31, 'classifier__min_samples_leaf': 10}


## Section 5: Comparison of Baseline vs. Optimized Performance

We tabulate the cross-validation performance comparison between baseline configurations and post-optimization models.

In [4]:
comparison_df = compare_with_baseline(candidates, optimization_results)
champion_name = select_production_candidate(comparison_df)
print(f"\nSelected Production Champion: {champion_name}\n")
comparison_df


Selected Production Champion: Logistic Regression



,model,baseline_cv_weighted_f1,optimized_cv_weighted_f1,best_params,improvement
0,Logistic Regression,0.756362,0.756344,"{'classifier__C': 0.75, 'classifier__max_iter'...",-0.000018
1,HistGradientBoosting,0.684177,0.687612,"{'classifier__l2_regularization': 10.0, 'class...",0.003435


## Section 6: Training the Champion Model on Full Training Set

With hyperparameter optimization finalized, we train the champion **Logistic Regression** pipeline on the complete training set ($N=1,600$).

In [5]:
champion_params = optimization_results[champion_name]["best_params"]
print(f"Training champion {champion_name} with parameters: {champion_params}")
champion_pipeline = train_final_pipeline(champion_name, champion_params, X_train, y_train)
print(f"Champion pipeline successfully fitted on {len(X_train)} training records.")

Training champion Logistic Regression with parameters: {'classifier__C': 0.75, 'classifier__max_iter': 500, 'classifier__solver': 'saga'}
Champion pipeline successfully fitted on 1600 training records.


## Section 7: Final Independent Test Set Evaluation

For the first and only time, we evaluate the finalized champion pipeline on the untouched test partition ($N=400$). All metrics reflect true out-of-sample generalization.

In [6]:
test_metrics, y_pred = evaluate_on_test_set(champion_pipeline, champion_name, X_test, y_test)
print(f"Final Test Accuracy:              {test_metrics['accuracy']:.4f}")
print(f"Final Test Weighted F1-score:     {test_metrics['weighted_f1']:.4f}")
print(f"Final Test Macro F1-score:        {test_metrics['macro_f1']:.4f}")
print(f"Final Test Weighted Precision:    {test_metrics['weighted_precision']:.4f}")
print(f"Final Test Weighted Recall:       {test_metrics['weighted_recall']:.4f}")
if "roc_auc_ovr" in test_metrics:
    print(f"Final Test Multi-Class ROC-AUC:   {test_metrics['roc_auc_ovr']:.4f}")

Final Test Accuracy:              0.7550
Final Test Weighted F1-score:     0.7557
Final Test Macro F1-score:        0.7551
Final Test Weighted Precision:    0.7565
Final Test Weighted Recall:       0.7550
Final Test Multi-Class ROC-AUC:   0.9129


## Section 8: Per-Class Diagnostic Report & Clinical Safety Audit

We inspect precision, recall, and F1-score across each individual mental health risk tier:
- **Low Risk:** Benign psychological state.
- **Medium Risk:** Moderate stress; monitoring advised.
- **High Risk:** Elevated distress; urgent clinical counseling required.

In [7]:
report_df = pd.DataFrame(test_metrics["classification_report"]).transpose()
report_df

,precision,recall,f1-score,support
High,0.810219,0.816176,0.813187,136.000
Low,0.828125,0.803030,0.815385,132.000
Medium,0.629630,0.643939,0.636704,132.000
accuracy,0.755000,0.755000,0.755000,0.755
macro avg,0.755991,0.754382,0.755092,400.000
weighted avg,0.756533,0.755000,0.755673,400.000


## Section 9: Confusion Matrix & Severe Misclassification Analysis

We visualize the $3 \times 3$ confusion matrix on the held-out test set ($N=400$) and verify error distributions.